In [2]:
import osmnx as ox

# Download and plot a street network
G = ox.graph_from_place("Curitiba, Parana, Brazil", network_type="drive")
# ox.plot_graph(G)

In [3]:
gdf_nodes, gdf_edges = ox.convert.graph_to_gdfs(G)

In [ ]:
import geopandas as gpd

# gdf_edges[['name','highway','geometry']].explore()

In [ ]:
from enum import Enum
import math
import geohexgrid as ghg
import numpy as np
from shapely.geometry import box
from shapely import Polygon, intersects

class GridFormat(Enum):
    SQUARE = "square"
    HEXAGON = "hexagon"

LOCATION = "Curitiba, Parana, Brazil"
GRID_FORMAT = GridFormat.HEXAGON
CELL_SIZE = 200

def _utm_crs(geom):
    """Retorna o EPSG UTM adequado para o centróide da geometria."""
    lon, lat = geom.centroid.x, geom.centroid.y
    zone = int((lon + 180) / 6) + 1
    south = lat < 0
    return f"EPSG:{32700 + zone if south else 32600 + zone}"

def _create_hexagon(length, center_x, center_y):
    """Creates a regular hexagon as a Shapely Polygon."""
    # A regular hexagon has 6 equal angles of 60 degrees
    angles = range(0, 360, 60)
    
    # Calculate the 6 vertices using trigonometry
    coords = [
        [center_x + math.cos(math.radians(angle)) * length,
         center_y + math.sin(math.radians(angle)) * length] 
        for angle in angles
    ]
    
    return Polygon(coords)

def generate_grid(location, grid_format, cell_size):
    city_gdf = ox.geocode_to_gdf(location)
    city_gdf = city_gdf.to_crs(_utm_crs(city_gdf.geometry.iloc[0]))
    
    minx, miny, maxx, maxy = city_gdf.total_bounds    

    if grid_format == GridFormat.HEXAGON:
        rows = math.ceil((maxy - miny) / (cell_size * math.sin(math.radians(60))))
        cols = math.ceil((maxx - minx) / (3 * cell_size))
        cells = []

        for i in range(rows):
            for j in range(cols):
                center_x = minx + j * 3 * cell_size + (1.5 * cell_size if i % 2 == 1 else 0)
                center_y = miny + i * cell_size * math.sin(math.radians(60))
                cells.append(_create_hexagon(cell_size, center_x, center_y))
    else:
        xs = np.arange(minx, maxx, cell_size)
        ys = np.arange(miny, maxy, cell_size)
        xx, yy = np.meshgrid(xs, ys)

        cells = [box(x, y, x + cell_size, y + cell_size) for x, y in zip(xx.ravel(), yy.ravel())]

    poly = city_gdf.geometry.iloc[0]
    mask = intersects(cells, poly)
    grid = gpd.GeoDataFrame(geometry=np.array(cells)[mask], crs=city_gdf.crs)

    grid = grid.to_crs("EPSG:4326")
    return grid

generate_grid(LOCATION, GRID_FORMAT, CELL_SIZE).to_file("curitiba_grid.shp")